# Tahap 02 — Data Cleaning dan Text Preprocessing

## Judul Project

**Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic**

## Tujuan Notebook

Notebook ini melanjutkan output dari Tahap 01 dan berfokus pada pembersihan serta preprocessing teks pidato secara aman dan terukur.

Tahap ini menghasilkan beberapa versi teks:

1. `text_clean_readable` — teks bersih yang tetap mudah dibaca, cocok untuk Manual Coding dan BERTopic embedding.
2. `text_clean_lower` — versi lowercase untuk kebutuhan analisis teks sederhana.
3. `tokens_basic` — token hasil preprocessing dasar untuk EDA token.
4. `text_preprocessed_tokens` — token yang digabung kembali menjadi teks, cocok untuk TF-IDF, CountVectorizer, atau EDA.
5. `text_for_manual_coding` — teks yang direkomendasikan untuk proses open coding, axial coding, dan selective coding.
6. `text_for_bertopic` — teks yang direkomendasikan untuk pemodelan BERTopic.

## Catatan

Notebook ini tidak mengasumsikan struktur data secara sembarangan. Setiap kolom yang digunakan akan divalidasi terlebih dahulu. Jika kolom wajib tidak tersedia, notebook akan berhenti dengan pesan error yang jelas.

## Output Tahap 02

Notebook ini menghasilkan file berikut:

```text
data/processed/speech_preprocessed_master.csv
reports/tables/text_preprocessing_summary.csv
reports/tables/top_tokens_by_speech.csv
reports/tables/preprocessing_quality_report.csv
reports/tables/stage02_output_manifest.json
```

In [1]:
# ============================================================
# Import Library
# ============================================================

from pathlib import Path
from datetime import datetime
from collections import Counter
import hashlib
import json
import re
import sys
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Library berhasil di-import.")
print(f"Python version : {sys.version}")
print(f"Pandas version : {pd.__version__}")

Library berhasil di-import.
Python version : 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Pandas version : 2.2.2


## 1. Setup Path Project

Cell ini mendeteksi root project secara otomatis dengan mencari file hasil Tahap 01, yaitu `data/interim/speech_raw_master.csv`. Jika notebook dijalankan dari folder `notebooks/`, root project tetap dapat ditemukan.

In [2]:
# ============================================================
# Setup Path Project
# ============================================================

def find_project_root(start_path=None):
    """
    Mendeteksi root project berdasarkan keberadaan output Tahap 01.
    Notebook akan mencari file data/interim/speech_raw_master.csv
    dari current directory hingga parent directory.
    """
    if start_path is None:
        start_path = Path.cwd().resolve()
    else:
        start_path = Path(start_path).resolve()

    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        required_file = candidate / "data" / "interim" / "speech_raw_master.csv"
        if required_file.exists():
            return candidate

    # fallback jika file Tahap 01 belum ditemukan
    if start_path.name.lower() == "notebooks":
        return start_path.parent

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLE_DIR = REPORTS_DIR / "tables"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

INPUT_RAW_MASTER_PATH = INTERIM_DIR / "speech_raw_master.csv"
INPUT_METADATA_PATH = INTERIM_DIR / "speech_metadata.csv"

OUTPUT_PREPROCESSED_MASTER_PATH = PROCESSED_DIR / "speech_preprocessed_master.csv"
OUTPUT_PREPROCESSING_SUMMARY_PATH = REPORT_TABLE_DIR / "text_preprocessing_summary.csv"
OUTPUT_TOP_TOKENS_PATH = REPORT_TABLE_DIR / "top_tokens_by_speech.csv"
OUTPUT_QUALITY_REPORT_PATH = REPORT_TABLE_DIR / "preprocessing_quality_report.csv"
OUTPUT_MANIFEST_PATH = REPORT_TABLE_DIR / "stage02_output_manifest.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project path berhasil disiapkan.")
print(f"Current working directory       : {Path.cwd().resolve()}")
print(f"PROJECT_ROOT                    : {PROJECT_ROOT}")
print(f"INPUT_RAW_MASTER_PATH           : {INPUT_RAW_MASTER_PATH}")
print(f"INPUT_METADATA_PATH             : {INPUT_METADATA_PATH}")
print(f"OUTPUT_PREPROCESSED_MASTER_PATH : {OUTPUT_PREPROCESSED_MASTER_PATH}")

Project path berhasil disiapkan.
Current working directory       : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\notebooks
PROJECT_ROOT                    : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
INPUT_RAW_MASTER_PATH           : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\speech_raw_master.csv
INPUT_METADATA_PATH             : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\speech_metadata.csv
OUTPUT_PREPROCESSED_MASTER_PATH : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_preprocessed_master.csv


## 2. Preflight Check Input Tahap 01

Cell ini memastikan file input dari Tahap 01 sudah tersedia sebelum proses cleaning dilakukan.

In [3]:
# ============================================================
# Preflight Check Input Tahap 01
# ============================================================

required_input_files = {
    "speech_raw_master": INPUT_RAW_MASTER_PATH,
    "speech_metadata": INPUT_METADATA_PATH,
}

missing_files = []
empty_files = []

for file_label, file_path in required_input_files.items():
    if not file_path.exists():
        missing_files.append(f"{file_label}: {file_path}")
    elif file_path.stat().st_size == 0:
        empty_files.append(f"{file_label}: {file_path}")

if missing_files:
    raise FileNotFoundError(
        "File input Tahap 01 berikut belum ditemukan:\n"
        + "\n".join(missing_files)
        + "\n\nJalankan Tahap 01 terlebih dahulu atau pastikan file berada di data/interim/."
    )

if empty_files:
    raise ValueError(
        "File input Tahap 01 berikut ditemukan tetapi kosong:\n"
        + "\n".join(empty_files)
    )

print("Preflight check berhasil. Seluruh file input Tahap 01 ditemukan dan tidak kosong.")

Preflight check berhasil. Seluruh file input Tahap 01 ditemukan dan tidak kosong.


## 3. Load Dataset dari Tahap 01

Dataset yang dibaca:

1. `speech_raw_master.csv` — berisi teks raw dan teks body.
2. `speech_metadata.csv` — berisi metadata dokumen pidato.

Setelah dibaca, notebook akan menampilkan daftar kolom aktual agar tidak ada asumsi struktur data yang tidak tervalidasi.

In [4]:
# ============================================================
# Load Dataset Tahap 01
# ============================================================

speech_raw_master_df = pd.read_csv(INPUT_RAW_MASTER_PATH)
speech_metadata_df = pd.read_csv(INPUT_METADATA_PATH)

print("Dataset berhasil dibaca.")
print(f"speech_raw_master_df shape : {speech_raw_master_df.shape}")
print(f"speech_metadata_df shape   : {speech_metadata_df.shape}")

print("\nKolom speech_raw_master_df:")
print(list(speech_raw_master_df.columns))

print("\nKolom speech_metadata_df:")
print(list(speech_metadata_df.columns))

Dataset berhasil dibaca.
speech_raw_master_df shape : (6, 23)
speech_metadata_df shape   : (6, 22)

Kolom speech_raw_master_df:
['speech_id', 'file_name', 'file_path', 'file_size_bytes', 'encoding_used', 'speech_title_from_filename', 'forum_scope_inferred', 'forum_scope_rule', 'event_date', 'event_date_source', 'source_url', 'source_domain', 'source_validation_status', 'url_count', 'language_estimate', 'text_raw', 'text_body', 'char_count_raw', 'char_count_body', 'word_count_body', 'line_count_body', 'content_sha256', 'processed_at']

Kolom speech_metadata_df:
['speech_id', 'file_name', 'speech_title_from_filename', 'forum_scope_inferred', 'forum_scope_rule', 'event_date', 'event_date_source', 'language_estimate', 'source_url', 'source_domain', 'source_validation_status', 'url_count', 'has_source_url', 'has_event_date', 'is_text_length_valid', 'file_size_bytes', 'char_count_body', 'word_count_body', 'line_count_body', 'content_sha256', 'quality_flags', 'processed_at']


## 4. Fungsi Validasi Schema

Cell ini menyediakan fungsi untuk memastikan kolom yang dibutuhkan benar-benar tersedia sebelum diproses.

In [5]:
# ============================================================
# Fungsi Validasi Schema
# ============================================================

def require_columns(df, required_columns, df_name="DataFrame"):
    """
    Memastikan DataFrame memiliki seluruh kolom wajib.
    Jika ada kolom yang tidak tersedia, proses dihentikan dengan pesan yang jelas.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{df_name} tidak memiliki kolom wajib: {missing_columns}. "
            f"Kolom tersedia: {list(df.columns)}"
        )


def get_existing_columns(df, candidate_columns):
    """
    Mengambil kolom yang benar-benar tersedia dari daftar kandidat kolom.
    Fungsi ini digunakan untuk kolom opsional agar tidak menimbulkan KeyError.
    """
    return [col for col in candidate_columns if col in df.columns]


RAW_REQUIRED_COLUMNS = [
    "speech_id",
    "file_name",
    "text_body"
]

METADATA_REQUIRED_COLUMNS = [
    "speech_id",
    "file_name"
]

require_columns(speech_raw_master_df, RAW_REQUIRED_COLUMNS, "speech_raw_master_df")
require_columns(speech_metadata_df, METADATA_REQUIRED_COLUMNS, "speech_metadata_df")

print("Validasi schema dasar berhasil.")

Validasi schema dasar berhasil.


## 5. Validasi Kualitas Dasar Input

Validasi yang dilakukan:

1. `speech_id` tidak boleh kosong.
2. `speech_id` harus unik.
3. `file_name` tidak boleh kosong.
4. `text_body` tidak boleh kosong.
5. Jumlah dokumen pada raw master dan metadata harus konsisten.

In [6]:
# ============================================================
# Validasi Kualitas Dasar Input
# ============================================================

if speech_raw_master_df["speech_id"].isna().any():
    raise ValueError("speech_raw_master_df memiliki speech_id kosong.")

if speech_raw_master_df["speech_id"].duplicated().any():
    duplicated_ids = speech_raw_master_df.loc[
        speech_raw_master_df["speech_id"].duplicated(keep=False),
        "speech_id"
    ].tolist()
    raise ValueError(f"speech_raw_master_df memiliki speech_id duplikat: {duplicated_ids}")

if speech_raw_master_df["file_name"].isna().any():
    raise ValueError("speech_raw_master_df memiliki file_name kosong.")

if speech_raw_master_df["text_body"].isna().any():
    raise ValueError("speech_raw_master_df memiliki text_body kosong.")

empty_text_rows = speech_raw_master_df[
    speech_raw_master_df["text_body"].astype(str).str.strip().eq("")
]

if not empty_text_rows.empty:
    display(empty_text_rows[["speech_id", "file_name"]])
    raise ValueError("Terdapat baris dengan text_body kosong setelah strip whitespace.")

if len(speech_raw_master_df) != len(speech_metadata_df):
    print("Peringatan: jumlah baris raw master dan metadata berbeda.")
    print(f"speech_raw_master_df : {len(speech_raw_master_df)}")
    print(f"speech_metadata_df   : {len(speech_metadata_df)}")
else:
    print("Jumlah baris raw master dan metadata konsisten.")

print("Validasi kualitas dasar input berhasil.")

Jumlah baris raw master dan metadata konsisten.
Validasi kualitas dasar input berhasil.


## 6. Parameter Preprocessing

Parameter dibuat eksplisit agar proses preprocessing dapat dijelaskan secara akademik dan dapat direplikasi.

Catatan penting untuk BERTopic:

- Untuk model berbasis embedding seperti BERTopic, teks tidak perlu dibersihkan secara terlalu agresif.
- Oleh karena itu, notebook tetap menyimpan `text_clean_readable` sebagai input utama untuk BERTopic.
- Token stopword removal digunakan untuk EDA token dan analisis berbasis frekuensi, bukan sebagai satu-satunya input BERTopic.

In [7]:
# ============================================================
# Parameter Preprocessing
# ============================================================

PREPROCESSING_CONFIG = {
    "remove_urls": True,
    "remove_email": True,
    "remove_source_note": True,
    "normalize_unicode": True,
    "normalize_whitespace": True,
    "lowercase_for_tokenization": True,
    "remove_numbers_for_tokens": True,
    "min_token_length": 3,
    "apply_general_stopwords": True,
    "apply_domain_stopwords": True,
    "stemming_enabled": False,
}

print("Konfigurasi preprocessing:")
print(json.dumps(PREPROCESSING_CONFIG, indent=2, ensure_ascii=False))

Konfigurasi preprocessing:
{
  "remove_urls": true,
  "remove_email": true,
  "remove_source_note": true,
  "normalize_unicode": true,
  "normalize_whitespace": true,
  "lowercase_for_tokenization": true,
  "remove_numbers_for_tokens": true,
  "min_token_length": 3,
  "apply_general_stopwords": true,
  "apply_domain_stopwords": true,
  "stemming_enabled": false
}


## 7. Stopwords Dasar Bahasa Indonesia dan Inggris

Dataset pidato memiliki campuran bahasa Indonesia dan Inggris. Karena itu, notebook menyediakan stopwords dasar untuk dua bahasa.

Stopwords domain juga disediakan untuk mengurangi kata-kata formal yang berulang pada hampir semua pidato, misalnya sapaan pembuka dan istilah administratif. Daftar ini dapat direvisi apabila pada tahap evaluasi ditemukan kata penting yang ikut terhapus.

In [8]:
# ============================================================
# Stopwords Dasar dan Stopwords Domain
# ============================================================

INDONESIAN_STOPWORDS = {
    "ada", "adalah", "agar", "akan", "akhir", "aku", "amat", "anda", "antara", "apa", "apabila",
    "atau", "atas", "awal", "bagai", "bagi", "bahwa", "baik", "bakal", "balik", "banyak", "baru",
    "bawah", "beberapa", "begini", "begitu", "belakang", "belum", "benar", "berada", "berakhir",
    "berikut", "bersama", "berturut", "bisa", "buat", "bukan", "cukup", "dalam", "dan", "dapat",
    "dari", "daripada", "dekat", "demi", "demikian", "dengan", "depan", "dia", "diberi", "diri",
    "dirinya", "dong", "dua", "dulu", "guna", "hal", "hanya", "hari", "harus", "hendak", "hingga",
    "ia", "ialah", "ibarat", "ibu", "ingin", "ini", "itu", "jadi", "jangan", "jauh", "jawab",
    "jelas", "jika", "juga", "justru", "kala", "kalau", "kami", "kamu", "kan", "kapan", "karena",
    "kata", "katakan", "kembali", "kemudian", "kepada", "ketika", "khusus", "kini", "kira", "kita",
    "kok", "lagi", "lah", "lain", "lalu", "lewat", "luar", "maka", "makin", "malah", "mampu",
    "mana", "masa", "masih", "masing", "mau", "maupun", "melalui", "memang", "mereka", "meski",
    "mungkin", "namun", "nanti", "nya", "oleh", "orang", "pada", "paling", "para", "perlu",
    "pernah", "pula", "pun", "saat", "saja", "saling", "sama", "sambil", "sampai", "sangat",
    "satu", "saya", "sebagai", "sebelum", "sebenarnya", "sebetulnya", "sebuah", "sehingga",
    "sekali", "sekalian", "sekitar", "selain", "selalu", "seluruh", "semakin", "sementara", "sempat",
    "semua", "sendiri", "seolah", "seperti", "sering", "serta", "siapa", "sini", "situ", "soal",
    "suatu", "sudah", "supaya", "tadi", "tanpa", "tapi", "telah", "tempat", "tentang", "tentu",
    "terhadap", "terima", "terus", "tetap", "tetapi", "tiap", "tidak", "tiga", "ujar", "untuk",
    "usah", "waktu", "wah", "ya", "yaitu", "yakni", "yang"
}

ENGLISH_STOPWORDS = {
    "about", "above", "after", "again", "against", "all", "also", "am", "an", "and", "any", "are",
    "as", "at", "be", "because", "been", "before", "being", "below", "between", "both", "but", "by",
    "can", "could", "did", "do", "does", "doing", "down", "during", "each", "few", "for", "from",
    "further", "had", "has", "have", "having", "he", "her", "here", "hers", "herself", "him", "himself",
    "his", "how", "i", "if", "in", "into", "is", "it", "its", "itself", "just", "me", "more",
    "most", "my", "myself", "no", "nor", "not", "now", "of", "off", "on", "once", "only", "or",
    "other", "our", "ours", "ourselves", "out", "over", "own", "same", "she", "should", "so", "some",
    "such", "than", "that", "the", "their", "theirs", "them", "themselves", "then", "there", "these",
    "they", "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "we",
    "were", "what", "when", "where", "which", "while", "who", "whom", "why", "will", "with", "you",
    "your", "yours", "yourself", "yourselves"
}

DOMAIN_STOPWORDS = {
    # Sapaan dan istilah protokoler yang sangat sering muncul
    "assalamualaikum", "bismillahirrahmanirrahim", "warahmatullahi", "wabarakatuh", "shalom", "syalom",
    "salve", "swastiastu", "namo", "buddhaya", "salam", "kebajikan", "rahayu", "wassalamualaikum",
    "om", "santi", "distinguished", "excellencies", "ladies", "gentlemen", "delegates", "friends",

    # Sapaan umum pidato
    "saudara", "saudara-saudara", "hadirin", "undangan", "hormat", "hormati", "banggakan",

    # Nama tokoh dan jabatan yang dapat terlalu dominan dalam seluruh naskah
    "prabowo", "presiden", "republik", "subianto",

    # Istilah administratif umum
    "menteri", "gubernur", "bupati", "wali", "kota", "kepala", "wakil"
}

print(f"Jumlah Indonesian stopwords : {len(INDONESIAN_STOPWORDS)}")
print(f"Jumlah English stopwords    : {len(ENGLISH_STOPWORDS)}")
print(f"Jumlah domain stopwords     : {len(DOMAIN_STOPWORDS)}")

Jumlah Indonesian stopwords : 189
Jumlah English stopwords    : 125
Jumlah domain stopwords     : 40


## 8. Fungsi Cleaning dan Tokenization

Fungsi dibuat modular agar mudah ditelusuri dan diuji ulang.

In [9]:
# ============================================================
# Fungsi Cleaning dan Tokenization
# ============================================================

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", flags=re.IGNORECASE)
READ_MORE_PATTERN = re.compile(r"^\s*read more\s*:\s*.*$", flags=re.IGNORECASE | re.MULTILINE)
MULTISPACE_PATTERN = re.compile(r"[ \t]+")
MULTINEWLINE_PATTERN = re.compile(r"\n{3,}")
TOKEN_PATTERN = re.compile(r"\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ'’\-]*\b", flags=re.UNICODE)


def normalize_unicode_text(text):
    """
    Menormalkan unicode agar karakter teks lebih konsisten.
    """
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    return text


def standardize_quotes_and_symbols(text):
    """
    Menstandarkan beberapa karakter kutip dan dash tanpa mengubah makna kalimat.
    """
    if not isinstance(text, str):
        return ""

    replacements = {
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        "—": "-",
        "–": "-",
        "‑": "-",
        "…": "...",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def remove_source_note(text):
    """
    Menghapus baris sumber seperti 'Read more: <url>'.
    """
    if not isinstance(text, str):
        return ""

    return READ_MORE_PATTERN.sub("", text)


def remove_urls_and_emails(text):
    """
    Menghapus URL dan email dari teks.
    """
    if not isinstance(text, str):
        return ""

    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)

    return text


def normalize_whitespace(text):
    """
    Menormalkan spasi dan baris kosong berlebih.
    """
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = MULTISPACE_PATTERN.sub(" ", text)
    text = MULTINEWLINE_PATTERN.sub("\n\n", text)

    return text.strip()


def clean_readable_text(text):
    """
    Membersihkan teks secara minimal agar tetap terbaca manusia.
    Cocok untuk manual coding dan BERTopic embedding.
    """
    if not isinstance(text, str):
        return ""

    cleaned = text

    if PREPROCESSING_CONFIG["normalize_unicode"]:
        cleaned = normalize_unicode_text(cleaned)

    cleaned = standardize_quotes_and_symbols(cleaned)

    if PREPROCESSING_CONFIG["remove_source_note"]:
        cleaned = remove_source_note(cleaned)

    if PREPROCESSING_CONFIG["remove_urls"] or PREPROCESSING_CONFIG["remove_email"]:
        cleaned = remove_urls_and_emails(cleaned)

    if PREPROCESSING_CONFIG["normalize_whitespace"]:
        cleaned = normalize_whitespace(cleaned)

    return cleaned


def tokenize_text(text, language_estimate="unknown"):
    """
    Tokenisasi teks dengan stopword removal dasar.
    Fungsi ini digunakan untuk EDA token, bukan menggantikan teks readable untuk BERTopic.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    text = normalize_unicode_text(text)
    text = standardize_quotes_and_symbols(text)
    text = text.lower()
    text = remove_urls_and_emails(text)
    text = remove_source_note(text)

    tokens = TOKEN_PATTERN.findall(text)
    tokens = [token.strip("'-").lower() for token in tokens]
    tokens = [token for token in tokens if token]

    min_len = PREPROCESSING_CONFIG["min_token_length"]
    tokens = [token for token in tokens if len(token) >= min_len]

    if PREPROCESSING_CONFIG["apply_general_stopwords"]:
        if language_estimate == "id":
            stopwords = INDONESIAN_STOPWORDS
        elif language_estimate == "en":
            stopwords = ENGLISH_STOPWORDS
        else:
            stopwords = INDONESIAN_STOPWORDS.union(ENGLISH_STOPWORDS)

        tokens = [token for token in tokens if token not in stopwords]

    if PREPROCESSING_CONFIG["apply_domain_stopwords"]:
        tokens = [token for token in tokens if token not in DOMAIN_STOPWORDS]

    return tokens


def count_words_regex(text):
    """
    Menghitung jumlah kata menggunakan regex sederhana.
    """
    if not isinstance(text, str) or not text.strip():
        return 0

    return len(TOKEN_PATTERN.findall(text))


def count_characters(text):
    """
    Menghitung jumlah karakter teks.
    """
    if not isinstance(text, str):
        return 0

    return len(text)


def create_text_hash(text):
    """
    Membuat SHA-256 hash dari teks untuk validasi perubahan konten.
    """
    if not isinstance(text, str):
        text = ""

    return hashlib.sha256(text.encode("utf-8")).hexdigest()


print("Fungsi cleaning dan tokenization berhasil dibuat.")

Fungsi cleaning dan tokenization berhasil dibuat.


## 9. Menentukan Kolom Metadata yang Akan Dibawa

Notebook hanya membawa kolom metadata yang benar-benar tersedia pada output Tahap 01. Kolom opsional tidak akan dipaksakan apabila tidak tersedia.

In [10]:
# ============================================================
# Menentukan Kolom Metadata Aktual
# ============================================================

metadata_candidate_columns = [
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "forum_scope_rule",
    "event_date",
    "event_date_source",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "quality_flags"
]

metadata_columns_to_merge = get_existing_columns(speech_metadata_df, metadata_candidate_columns)

print("Kolom metadata yang akan digunakan:")
print(metadata_columns_to_merge)

metadata_for_merge_df = speech_metadata_df[metadata_columns_to_merge].copy()

Kolom metadata yang akan digunakan:
['speech_id', 'file_name', 'speech_title_from_filename', 'forum_scope_inferred', 'forum_scope_rule', 'event_date', 'event_date_source', 'language_estimate', 'source_url', 'source_domain', 'source_validation_status', 'quality_flags']


## 10. Membuat Dataset Preprocessed Master

Cell ini membuat dataset utama hasil preprocessing. Proses dilakukan pada level dokumen pidato, belum pada level chunk. Chunking akan dilakukan pada tahap berikutnya.

In [11]:
# ============================================================
# Membuat Dataset Preprocessed Master
# ============================================================

base_columns = ["speech_id", "file_name", "text_body"]
require_columns(speech_raw_master_df, base_columns, "speech_raw_master_df")

# Kolom tambahan dari raw master yang dibawa jika tersedia
raw_optional_columns = [
    "file_size_bytes",
    "encoding_used",
    "char_count_body",
    "word_count_body",
    "line_count_body",
    "content_sha256"
]

raw_columns_to_keep = base_columns + get_existing_columns(speech_raw_master_df, raw_optional_columns)

speech_preprocessed_df = speech_raw_master_df[raw_columns_to_keep].copy()

# Merge metadata jika tersedia
metadata_non_key_columns = [col for col in metadata_columns_to_merge if col not in {"speech_id", "file_name"}]

if metadata_non_key_columns:
    speech_preprocessed_df = speech_preprocessed_df.merge(
        metadata_for_merge_df[["speech_id", "file_name"] + metadata_non_key_columns],
        on=["speech_id", "file_name"],
        how="left",
        validate="one_to_one"
    )

# Cleaning readable text
speech_preprocessed_df["text_clean_readable"] = speech_preprocessed_df["text_body"].apply(clean_readable_text)
speech_preprocessed_df["text_clean_lower"] = speech_preprocessed_df["text_clean_readable"].str.lower()

# Pastikan language_estimate tersedia untuk tokenisasi
if "language_estimate" not in speech_preprocessed_df.columns:
    speech_preprocessed_df["language_estimate"] = "unknown"
else:
    speech_preprocessed_df["language_estimate"] = speech_preprocessed_df["language_estimate"].fillna("unknown")

# Tokenisasi
speech_preprocessed_df["tokens_basic"] = speech_preprocessed_df.apply(
    lambda row: tokenize_text(row["text_clean_readable"], row["language_estimate"]),
    axis=1
)

speech_preprocessed_df["text_preprocessed_tokens"] = speech_preprocessed_df["tokens_basic"].apply(lambda tokens: " ".join(tokens))

# Kolom rekomendasi penggunaan tahap berikutnya
speech_preprocessed_df["text_for_manual_coding"] = speech_preprocessed_df["text_clean_readable"]
speech_preprocessed_df["text_for_bertopic"] = speech_preprocessed_df["text_clean_readable"]

# Statistik setelah preprocessing
speech_preprocessed_df["char_count_clean"] = speech_preprocessed_df["text_clean_readable"].apply(count_characters)
speech_preprocessed_df["word_count_clean"] = speech_preprocessed_df["text_clean_readable"].apply(count_words_regex)
speech_preprocessed_df["token_count_basic"] = speech_preprocessed_df["tokens_basic"].apply(len)
speech_preprocessed_df["unique_token_count_basic"] = speech_preprocessed_df["tokens_basic"].apply(lambda tokens: len(set(tokens)))
speech_preprocessed_df["text_clean_sha256"] = speech_preprocessed_df["text_clean_readable"].apply(create_text_hash)

speech_preprocessed_df["preprocessing_config_json"] = json.dumps(PREPROCESSING_CONFIG, ensure_ascii=False)
speech_preprocessed_df["processed_stage02_at"] = datetime.now().isoformat(timespec="seconds")

print(f"Dataset preprocessed berhasil dibuat dengan shape: {speech_preprocessed_df.shape}")
display(speech_preprocessed_df.head())

Dataset preprocessed berhasil dibuat dengan shape: (6, 32)


,speech_id,file_name,text_body,file_size_bytes,encoding_used,char_count_body,word_count_body,line_count_body,content_sha256,speech_title_from_filename,forum_scope_inferred,forum_scope_rule,event_date,event_date_source,language_estimate,source_url,source_domain,source_validation_status,quality_flags,text_clean_readable,text_clean_lower,tokens_basic,text_preprocessed_tokens,text_for_manual_coding,text_for_bertopic,char_count_clean,word_count_clean,token_count_basic,unique_token_count_basic,text_clean_sha256,preprocessing_config_json,processed_stage02_at
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,1888,utf-8,1642,256,8,a2f66ccd4a1d6ba1b4793089989eb09607add274f280505c062aa477bdaface9,Brics Leaders,international,filename_contains_international_forum_keyword,2025-09-08,source_url_or_text_pattern,en,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-brics-leaders-virtual-meeting-melalui-video-conferenc...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,OK,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,distinguished leaders of brics.\n\nit is indeed a great honor for me to join this very important meeting.\n\nindones...,"[leaders, brics, indeed, great, honor, join, important, meeting, indonesia, considers, brics, strong, pillar, stabil...",leaders brics indeed great honor join important meeting indonesia considers brics strong pillar stability hope curre...,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,1642,254,138,101,8cd52135a957c95201d1430d1730dba226a5309a51ede801064fd3290daf8e21,"{""remove_urls"": true, ""remove_email"": true, ""remove_source_note"": true, ""normalize_unicode"": true, ""normalize_whites...",2026-06-11T21:38:26
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,"Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...",24596,utf-8,24220,3417,82,2fe161589989ad5494b22841bb0c3bc26977e64fecbd0c9994e413e47e42db77,Panen Raya,national,filename_contains_national_event_keyword,2026-01-07,source_url_or_text_pattern,id,https://setkab.go.id/sambutan-presiden-republik-indonesia-pada-panen-raya-dan-pengumuman-swasembada-pangan-di-desa-k...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,OK,"Bismillahirrahmanirrahim.\n\nAssalamu'alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...","bismillahirrahmanirrahim.\n\nassalamu'alaikum warahmatullahi wabarakatuh,\nselamat siang,\nsalam sejahtera bagi kita...","[assalamu'alaikum, selamat, siang, sejahtera, pertanian, penyelenggara, andi, amran, sulaiman, beserta, jajaran, kem...",assalamu'alaikum selamat siang sejahtera pertanian penyelenggara andi amran sulaiman beserta jajaran kementerian per...,"Bismillahirrahmanirrahim.\n\nAssalamu'alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...","Bismillahirrahmanirrahim.\n\nAssalamu'alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...",24220,3388,1701,809,26334373815c3bc7667d520e4bb95f1a94aaa7a2620107cd580829bbdd17c955,"{""remove_urls"": true, ""remove_email"": true, ""remove_source_note"": true, ""normalize_unicode"": true, ""normalize_whites...",2026-06-11T21:38:26
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,"Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat sore,\nSalam sejahtera bagi kita ...",18090,utf-8,17990,2543,54,881ebf65c5148e942e7fb142ee7950d6d91a5381cdbb4683c7b96a3beb024fda,Peresmian Infrastruktur Energi,national,filename_contains_national_event_keyword,NaN,date_not_found,id,NaN,NaN,MISSING_S

## 11. Validasi Output Preprocessing

Validasi yang dilakukan:

1. Jumlah baris output harus sama dengan jumlah input.
2. `speech_id` tetap unik.
3. `text_clean_readable` tidak boleh kosong.
4. `text_for_bertopic` tidak boleh kosong.
5. Token hasil preprocessing tidak boleh kosong.
6. Tidak ada duplikasi hash teks bersih.

In [12]:
# ============================================================
# Validasi Output Preprocessing
# ============================================================

PREPROCESSED_REQUIRED_COLUMNS = [
    "speech_id",
    "file_name",
    "text_body",
    "text_clean_readable",
    "text_clean_lower",
    "tokens_basic",
    "text_preprocessed_tokens",
    "text_for_manual_coding",
    "text_for_bertopic",
    "char_count_clean",
    "word_count_clean",
    "token_count_basic",
    "unique_token_count_basic",
    "text_clean_sha256"
]

require_columns(speech_preprocessed_df, PREPROCESSED_REQUIRED_COLUMNS, "speech_preprocessed_df")

if len(speech_preprocessed_df) != len(speech_raw_master_df):
    raise ValueError("Jumlah baris output preprocessing tidak sama dengan input raw master.")

if speech_preprocessed_df["speech_id"].duplicated().any():
    duplicated_ids = speech_preprocessed_df.loc[
        speech_preprocessed_df["speech_id"].duplicated(keep=False),
        "speech_id"
    ].tolist()
    raise ValueError(f"Terdapat speech_id duplikat setelah preprocessing: {duplicated_ids}")

empty_clean_df = speech_preprocessed_df[
    speech_preprocessed_df["text_clean_readable"].astype(str).str.strip().eq("")
]

if not empty_clean_df.empty:
    display(empty_clean_df[["speech_id", "file_name"]])
    raise ValueError("Terdapat text_clean_readable kosong.")

empty_bertopic_df = speech_preprocessed_df[
    speech_preprocessed_df["text_for_bertopic"].astype(str).str.strip().eq("")
]

if not empty_bertopic_df.empty:
    display(empty_bertopic_df[["speech_id", "file_name"]])
    raise ValueError("Terdapat text_for_bertopic kosong.")

empty_token_df = speech_preprocessed_df[
    speech_preprocessed_df["token_count_basic"] <= 0
]

if not empty_token_df.empty:
    display(empty_token_df[["speech_id", "file_name", "token_count_basic"]])
    raise ValueError("Terdapat dokumen tanpa token setelah preprocessing.")

duplicate_clean_hash_df = speech_preprocessed_df[
    speech_preprocessed_df["text_clean_sha256"].duplicated(keep=False)
]

if not duplicate_clean_hash_df.empty:
    print("Peringatan: terdapat potensi duplikasi text_clean_readable berdasarkan hash.")
    display(duplicate_clean_hash_df[["speech_id", "file_name", "text_clean_sha256"]])
else:
    print("Tidak ditemukan duplikasi text_clean_readable berdasarkan hash.")

print("Validasi output preprocessing berhasil.")

Tidak ditemukan duplikasi text_clean_readable berdasarkan hash.
Validasi output preprocessing berhasil.


## 12. Membuat Ringkasan Preprocessing

Ringkasan ini membandingkan panjang teks sebelum dan sesudah cleaning.

In [13]:
# ============================================================
# Membuat Text Preprocessing Summary
# ============================================================

summary_records = []

for _, row in speech_preprocessed_df.iterrows():
    original_word_count = row["word_count_body"] if "word_count_body" in speech_preprocessed_df.columns else count_words_regex(row["text_body"])
    original_char_count = row["char_count_body"] if "char_count_body" in speech_preprocessed_df.columns else count_characters(row["text_body"])

    word_reduction = original_word_count - row["word_count_clean"]
    char_reduction = original_char_count - row["char_count_clean"]

    summary_records.append({
        "speech_id": row["speech_id"],
        "file_name": row["file_name"],
        "language_estimate": row.get("language_estimate", "unknown"),
        "word_count_before": int(original_word_count),
        "word_count_after_clean": int(row["word_count_clean"]),
        "word_count_difference": int(word_reduction),
        "char_count_before": int(original_char_count),
        "char_count_after_clean": int(row["char_count_clean"]),
        "char_count_difference": int(char_reduction),
        "token_count_basic": int(row["token_count_basic"]),
        "unique_token_count_basic": int(row["unique_token_count_basic"]),
    })

text_preprocessing_summary_df = pd.DataFrame(summary_records)

display(text_preprocessing_summary_df)

,speech_id,file_name,language_estimate,word_count_before,word_count_after_clean,word_count_difference,char_count_before,char_count_after_clean,char_count_difference,token_count_basic,unique_token_count_basic
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,256,254,2,1642,1642,0,138,101
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,id,3417,3388,29,24220,24220,0,1701,809
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,id,2543,2507,36,17990,17992,-2,1349,708
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,id,3484,3441,43,24459,24461,-2,1715,872
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,en,3585,3495,90,21135,21135,0,1728,854
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,en,1824,1814,10,11116,11116,0,977,566


## 13. Membuat Top Tokens per Pidato

Tabel ini digunakan untuk EDA awal dan membantu melihat kata-kata dominan setelah stopword removal.

In [14]:
# ============================================================
# Membuat Top Tokens per Pidato
# ============================================================

TOP_N_TOKENS_PER_SPEECH = 25

top_token_records = []

for _, row in speech_preprocessed_df.iterrows():
    token_counter = Counter(row["tokens_basic"])

    for rank, (token, frequency) in enumerate(token_counter.most_common(TOP_N_TOKENS_PER_SPEECH), start=1):
        top_token_records.append({
            "speech_id": row["speech_id"],
            "file_name": row["file_name"],
            "language_estimate": row.get("language_estimate", "unknown"),
            "rank": rank,
            "token": token,
            "frequency": frequency
        })

top_tokens_by_speech_df = pd.DataFrame(top_token_records)

display(top_tokens_by_speech_df.head(50))
print(f"Jumlah baris top token: {len(top_tokens_by_speech_df)}")

,speech_id,file_name,language_estimate,rank,token,frequency
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,1,brics,7
1,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,2,must,5
2,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,3,world,4
3,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,4,president,4
4,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,5,countries,4
5,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,6,indonesia,3
6,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,7,fully,3
7,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,8,support,3
8,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,9,biggest,3
9,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,10,continue,3


Jumlah baris top token: 150


## 14. Membuat Quality Report

Quality report digunakan untuk menandai potensi masalah setelah preprocessing, misalnya teks terlalu pendek atau jumlah token terlalu sedikit.

In [15]:
# ============================================================
# Membuat Preprocessing Quality Report
# ============================================================

def build_preprocessing_quality_flags(row):
    flags = []

    if row["word_count_clean"] < 100:
        flags.append("LOW_CLEAN_WORD_COUNT_LT_100")

    if row["token_count_basic"] < 50:
        flags.append("LOW_TOKEN_COUNT_LT_50")

    if row["unique_token_count_basic"] < 30:
        flags.append("LOW_UNIQUE_TOKEN_COUNT_LT_30")

    if row["char_count_clean"] <= 0:
        flags.append("EMPTY_CLEAN_TEXT")

    if not flags:
        return "OK"

    return ";".join(flags)


preprocessing_quality_report_df = speech_preprocessed_df.copy()
preprocessing_quality_report_df["preprocessing_quality_flags"] = preprocessing_quality_report_df.apply(
    build_preprocessing_quality_flags,
    axis=1
)

quality_report_columns = [
    "speech_id",
    "file_name",
    "language_estimate",
    "word_count_clean",
    "char_count_clean",
    "token_count_basic",
    "unique_token_count_basic",
    "preprocessing_quality_flags"
]

quality_report_columns = get_existing_columns(preprocessing_quality_report_df, quality_report_columns)
preprocessing_quality_report_df = preprocessing_quality_report_df[quality_report_columns].copy()

display(preprocessing_quality_report_df)

print("Ringkasan preprocessing_quality_flags:")
display(
    preprocessing_quality_report_df["preprocessing_quality_flags"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "preprocessing_quality_flags", "preprocessing_quality_flags": "document_count"})
)

,speech_id,file_name,language_estimate,word_count_clean,char_count_clean,token_count_basic,unique_token_count_basic,preprocessing_quality_flags
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,en,254,1642,138,101,OK
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,id,3388,24220,1701,809,OK
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,id,2507,17992,1349,708,OK
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,id,3441,24461,1715,872,OK
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,en,3495,21135,1728,854,OK
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,en,1814,11116,977,566,OK


Ringkasan preprocessing_quality_flags:


,document_count,count
0,OK,6


## 15. Preview Teks Sebelum dan Sesudah Cleaning

Cell ini menampilkan potongan teks untuk memastikan hasil cleaning tetap masuk akal dan tidak merusak substansi pidato.

In [16]:
# ============================================================
# Preview Teks Sebelum dan Sesudah Cleaning
# ============================================================

PREVIEW_CHAR_LIMIT = 600

preview_records = []

for _, row in speech_preprocessed_df.iterrows():
    preview_records.append({
        "speech_id": row["speech_id"],
        "file_name": row["file_name"],
        "text_body_preview": str(row["text_body"])[:PREVIEW_CHAR_LIMIT],
        "text_clean_readable_preview": str(row["text_clean_readable"])[:PREVIEW_CHAR_LIMIT],
        "text_preprocessed_tokens_preview": str(row["text_preprocessed_tokens"])[:PREVIEW_CHAR_LIMIT]
    })

text_preview_df = pd.DataFrame(preview_records)
display(text_preview_df)

,speech_id,file_name,text_body_preview,text_clean_readable_preview,text_preprocessed_tokens_preview
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,Distinguished Leaders of BRICS.\n\nIt is indeed a great honor for me to join this very important meeting.\n\nIndones...,leaders brics indeed great honor join important meeting indonesia considers brics strong pillar stability hope curre...
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,"Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...","Bismillahirrahmanirrahim.\n\nAssalamu'alaikum warahmatullahi wabarakatuh,\nSelamat siang,\nSalam sejahtera bagi kita...",assalamu'alaikum selamat siang sejahtera pertanian penyelenggara andi amran sulaiman beserta jajaran kementerian per...
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-ENERGI.txt,"Bismillahirrahmanirrahim.\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat sore,\nSalam sejahtera bagi kita ...","Bismillahirrahmanirrahim.\n\nAssalamu'alaikum warahmatullahi wabarakatuh,\nSelamat sore,\nSalam sejahtera bagi kita ...",assalamu'alaikum selamat sore sejahtera energi sumber daya mineral esdm bahlil lahadalia direktur utama pertamina si...
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,"Bismillahirrahmanirrahim,\n\nAssalamu’alaikum warahmatullahi wabarakatuh,\nSelamat pagi oh sudah selamat siang,\nSal...","Bismillahirrahmanirrahim,\n\nAssalamu'alaikum warahmatullahi wabarakatuh,\nSelamat pagi oh sudah selamat siang,\nSal...",assalamu'alaikum selamat pagi selamat siang sejahtera sosial saifullah yusuf penyelenggara sosial agus jabo priyono ...
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,"Distinguished President and CEO of the WEF, Mr. Borge Brende,\n\nLadies and Gentlemen,\n\nBismillahirrahmanirrahim,\...","Distinguished President and CEO of the WEF, Mr. Borge Brende,\n\nLadies and Gentlemen,\n\nBismillahirrahmanirrahim,\...",president ceo wef borge brende assalamu'alaikum gather davos time great uncertainty time wars continue break time tr...
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,"His Excellency Mr. Antoni Guterres, Secretary General of the United Nations;\nHer Excellency Madam Annalena Baerbock...","His Excellency Mr. Antoni Guterres, Secretary General of the United Nations;\nHer Excellency Madam Annalena Baerbock...",excellency antoni guterres secretary general united nations excellency madam annalena baerbock president united nati...


## 16. Validasi Akhir Sebelum Menyimpan Output

Sebelum output disimpan, notebook memastikan semua DataFrame output memiliki struktur yang valid.

In [17]:
# ============================================================
# Validasi Akhir Sebelum Save
# ============================================================

# Validasi output utama
require_columns(speech_preprocessed_df, PREPROCESSED_REQUIRED_COLUMNS, "speech_preprocessed_df")
require_columns(text_preprocessing_summary_df, ["speech_id", "file_name", "word_count_before", "word_count_after_clean"], "text_preprocessing_summary_df")
require_columns(top_tokens_by_speech_df, ["speech_id", "file_name", "rank", "token", "frequency"], "top_tokens_by_speech_df")
require_columns(preprocessing_quality_report_df, ["speech_id", "file_name", "preprocessing_quality_flags"], "preprocessing_quality_report_df")

if len(speech_preprocessed_df) == 0:
    raise ValueError("speech_preprocessed_df kosong.")

if len(text_preprocessing_summary_df) != len(speech_preprocessed_df):
    raise ValueError("Jumlah baris summary tidak sama dengan jumlah dokumen preprocessed.")

if len(preprocessing_quality_report_df) != len(speech_preprocessed_df):
    raise ValueError("Jumlah baris quality report tidak sama dengan jumlah dokumen preprocessed.")

print("Validasi akhir berhasil. Output siap disimpan.")

Validasi akhir berhasil. Output siap disimpan.


## 17. Menyimpan Output Tahap 02

Output akan disimpan ke folder `data/processed/` dan `reports/tables/`.

In [18]:
# ============================================================
# Save Output Tahap 02
# ============================================================

# Agar list token aman disimpan ke CSV, tokens_basic dikonversi ke JSON string.
speech_preprocessed_export_df = speech_preprocessed_df.copy()
speech_preprocessed_export_df["tokens_basic"] = speech_preprocessed_export_df["tokens_basic"].apply(
    lambda tokens: json.dumps(tokens, ensure_ascii=False)
)

speech_preprocessed_export_df.to_csv(
    OUTPUT_PREPROCESSED_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

text_preprocessing_summary_df.to_csv(
    OUTPUT_PREPROCESSING_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

top_tokens_by_speech_df.to_csv(
    OUTPUT_TOP_TOKENS_PATH,
    index=False,
    encoding="utf-8-sig"
)

preprocessing_quality_report_df.to_csv(
    OUTPUT_QUALITY_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Output Tahap 02 berhasil disimpan:")
print(f"1. {OUTPUT_PREPROCESSED_MASTER_PATH}")
print(f"2. {OUTPUT_PREPROCESSING_SUMMARY_PATH}")
print(f"3. {OUTPUT_TOP_TOKENS_PATH}")
print(f"4. {OUTPUT_QUALITY_REPORT_PATH}")

Output Tahap 02 berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_preprocessed_master.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\text_preprocessing_summary.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\top_tokens_by_speech.csv
4. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\preprocessing_quality_report.csv


## 18. Membuat Manifest Output

Manifest digunakan untuk mendokumentasikan file yang dihasilkan, parameter preprocessing, dan waktu proses.

In [19]:
# ============================================================
# Membuat Manifest Output Tahap 02
# ============================================================

manifest = {
    "stage": "02_text_cleaning_preprocessing",
    "project_title": "Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "input_files": {
        "speech_raw_master": str(INPUT_RAW_MASTER_PATH),
        "speech_metadata": str(INPUT_METADATA_PATH),
    },
    "output_files": {
        "speech_preprocessed_master": str(OUTPUT_PREPROCESSED_MASTER_PATH),
        "text_preprocessing_summary": str(OUTPUT_PREPROCESSING_SUMMARY_PATH),
        "top_tokens_by_speech": str(OUTPUT_TOP_TOKENS_PATH),
        "preprocessing_quality_report": str(OUTPUT_QUALITY_REPORT_PATH),
    },
    "document_count": int(len(speech_preprocessed_df)),
    "preprocessing_config": PREPROCESSING_CONFIG,
    "columns_output_master": list(speech_preprocessed_export_df.columns),
}

with open(OUTPUT_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

print(f"Manifest output berhasil disimpan ke: {OUTPUT_MANIFEST_PATH}")
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:1500])

Manifest output berhasil disimpan ke: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\stage02_output_manifest.json
{
  "stage": "02_text_cleaning_preprocessing",
  "project_title": "Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic",
  "generated_at": "2026-06-11T21:39:31",
  "project_root": "D:\\DATA-KAMIL\\MATKUL\\SEMESTER-1\\DATA-MINING\\topic-modeling-prabowo-un-speech",
  "input_files": {
    "speech_raw_master": "D:\\DATA-KAMIL\\MATKUL\\SEMESTER-1\\DATA-MINING\\topic-modeling-prabowo-un-speech\\data\\interim\\speech_raw_master.csv",
    "speech_metadata": "D:\\DATA-KAMIL\\MATKUL\\SEMESTER-1\\DATA-MINING\\topic-modeling-prabowo-un-speech\\data\\interim\\speech_metadata.csv"
  },
  "output_files": {
    "speech_preprocessed_master": "D:\\DATA-KAMIL\\MATKUL\\SEMESTER-1\\DATA-MINING\\topic-modeling-prabowo-un-speech\\data\\processed\\speech_preprocessed_master.c

## 19. Kesimpulan Tahap 02

Tahap 02 telah menghasilkan dataset teks yang lebih bersih dan siap digunakan untuk tahap berikutnya.

Output utama yang digunakan pada tahap berikutnya adalah:

```text
data/processed/speech_preprocessed_master.csv
```

Kolom yang direkomendasikan:

1. Untuk Manual Coding: `text_for_manual_coding`
2. Untuk BERTopic: `text_for_bertopic`
3. Untuk EDA token: `text_preprocessed_tokens`